# Modelo final (R09)

Mede o modelo escolhido no conjunto de teste, uma única vez, e o converte para o celular (TFLite). Não treina nada: processa só as imagens da validação e do teste (~7 mil), escolhe os limiares na validação e os aplica ao teste, nos três modos de TTA. Depois gera os arquivos TFLite (float32, float16 e int8) em `modelos/tflite/` e confere que preveem o mesmo que o modelo Keras.

A lógica está em `src/retinopatia/modelo_final.py`. A sessão desliga sozinha no fim.

## Preparação do repositório

In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

# Versão do código a usar. "origin/main" é a mais recente no GitHub.
# Para reproduzir um treino antigo, use a tag ou o commit dele
# (por exemplo "treino-v2"); o commit de cada treino fica no
# configuracao.json da pasta de resultados e no registro de experimentos.
REVISAO_CODIGO = "origin/v4-experimentos"

# Final: o modelo escolhido (CAMINHO_MODELO_ESCOLHIDO em configuracao.py)
# medido no teste e convertido para TFLite. Sem treino.
EXPERIMENTO = "R09_final_b0_768"
os.environ["RETINOPATIA_EXPERIMENTO"] = EXPERIMENTO

os.environ["RETINOPATIA_FUMACA"] = "0"

# Abre o conjunto de teste. Rode este notebook UMA VEZ SÓ: medir o teste
# de novo depois de mexer em algo transforma o teste em validação.
os.environ["RETINOPATIA_AVALIAR_TESTE"] = "1"

# True = encerra a sessão no fim, também se der erro
DESLIGAR_AO_TERMINAR = True

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}

!git -C {PASTA_REPOSITORIO} fetch --tags origin
!git -C {PASTA_REPOSITORIO} checkout --detach {REVISAO_CODIGO}
!git -C {PASTA_REPOSITORIO} log -1 --date=short --format="Código no commit %h (%cd): %s"

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente quando esta célula traz código
# novo, evitando ter que reiniciar o ambiente a cada edição no VS Code.
# O Colab usa Python 3.12+, que não tem mais o módulo "imp", mas a
# versão do IPython instalada nele ainda o importa no autoreload.
# Um substituto mínimo resolve; se mesmo assim falhar, o notebook
# segue sem autoreload (aí é preciso reiniciar após cada git pull).
import importlib
import types

try:
    import imp
except ModuleNotFoundError:
    sys.modules["imp"] = types.ModuleType("imp")
    sys.modules["imp"].reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as erro:
    print("Autoreload indisponível:", erro)

print("Repositório pronto em:", PASTA_REPOSITORIO)

## Conexão com o Drive

In [ ]:
from retinopatia import ambiente_colab, modelo_final

ambiente_colab.montar_drive()

## Teste e TFLite

In [ ]:
resultado_validacao, resultado_teste, tabela_tflite = modelo_final.rodar_final(
    desligar_ao_terminar=DESLIGAR_AO_TERMINAR
)